In [15]:
import pandas as pd
import numpy as np
import sys
sys.path.append(r"C:\Users\bunyo\OneDrive\Desktop\AI_Course\ModularProgramProjects\finalProject")
from src.data_preprocessing import Preprocessor

In [16]:
df = pd.read_csv(r"C:\Users\bunyo\OneDrive\Desktop\AI_Course\ModularProgramProjects\finalProject\data\raw_data\cleaned_raw_data.csv")

In [17]:
df.sample(2)

,Name,Range_km,Efficiency,Weight_kg,Acceleration(0-100),Firth_stop_range_km,Battery_kWh,Fast_charger(kW),Towing_kg,Cargo_volume,Price/Range(km),Price_Euro
166,Tesla Model 3 Long Range AWD,465,155,1919,4.4,534,72.0,108,1000,649,121,53.56
481,Volvo XC40 Recharge Pure Electric,330,203,2023,7.4,355,67.0,113,1500,450,146,45.08


In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Name                 1200 non-null   str    
 1   Range_km             1200 non-null   int64  
 2   Efficiency           1200 non-null   int64  
 3   Weight_kg            1200 non-null   int64  
 4   Acceleration(0-100)  1200 non-null   float64
 5   Firth_stop_range_km  1200 non-null   int64  
 6   Battery_kWh          1200 non-null   float64
 7   Fast_charger(kW)     1200 non-null   int64  
 8   Towing_kg            1200 non-null   int64  
 9   Cargo_volume         1200 non-null   int64  
 10  Price/Range(km)      1200 non-null   int64  
 11  Price_Euro           1200 non-null   float64
dtypes: float64(3), int64(8), str(1)
memory usage: 112.6 KB


# preprocessing

In [19]:
df['Range_km_level'] = pd.cut(df['Range_km'],bins=[0, 340, 440, np.inf], labels=['short','middle','long'])
df['Price_level'] = pd.cut(df['Price_Euro'], bins=[0, 45, 64, np.inf], labels=['cheap', 'affordable', 'expensive'])

In [20]:
df = df.drop(columns=['Price_Euro','Range_km','Name'])

In [21]:
mapping = {
    'Price_level' : ['cheap','affordable','expensive'],
    'Range_km_level' : ['short','middle','long']
           }

target = ['Price_level','Range_km_level']

In [22]:
result = Preprocessor(df)
pre_data = (
    result.fill_nan()
    # .ordinal_encoding(mapping=mapping)
    .scaler(target = target)
    .get_preprocessed_data()
)

In [23]:
pre_data['Range_km_level'] = pre_data['Range_km_level'].replace(['short','middle','long'],[0,1,2])
pre_data['Price_level'] = pre_data['Price_level'].replace(['cheap','affordable','expensive'],[0,1,2])

In [24]:
pre_data.head()

,Efficiency,Weight_kg,Acceleration(0-100),Firth_stop_range_km,Battery_kWh,Fast_charger(kW),Towing_kg,Cargo_volume,Price/Range(km),Range_km_level,Price_level
0,0.370968,0.650771,0.142157,0.878080,0.868203,0.696970,0.666667,0.2890,0.042895,2,2
1,0.258065,0.454721,0.254902,0.660182,0.576037,0.393939,0.533333,0.4760,0.024129,2,1
2,0.193548,0.503854,0.230392,0.883268,0.649770,0.630303,0.500000,0.2530,0.013405,2,1
3,0.333333,0.345376,0.289216,0.435798,0.435023,0.324242,0.166667,0.1815,0.028150,1,0
4,0.139785,0.403661,0.205882,0.594034,0.419355,0.333333,0.333333,0.3410,0.009383,2,0


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [26]:
x = pre_data.drop(columns=['Range_km_level','Price_level'])
y = pre_data[['Range_km_level','Price_level']].astype(int)

In [27]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.19, random_state=99)

In [39]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Extra Trees": ExtraTreesClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "Logistic Regression": MultiOutputClassifier(LogisticRegression(max_iter=1000)),
    "SVM (SVC)": MultiOutputClassifier(SVC(random_state=42)),
    "Gradient Boosting": MultiOutputClassifier(GradientBoostingClassifier(random_state=42)),
    "XGBoost": MultiOutputClassifier(XGBClassifier(random_state=42, eval_metric='mlogloss')),
}

results = []
for name, model in models.items():
    try:
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)
        
        acc_price = accuracy_score(y_test.iloc[:, 0], y_pred[:, 0])
        acc_range = accuracy_score(y_test.iloc[:, 1], y_pred[:, 1])
        avg_acc = (acc_price + acc_range) / 2
        
        results.append({
            "Model": name,
            "Price Accuracy (%)": round(acc_price * 100, 2),
            "Range Accuracy (%)": round(acc_range * 100, 2),
            "Average": round(avg_acc * 100, 2)
        })
    except Exception as e:
        print(f"Errors: {e}")

results_df = pd.DataFrame(results).sort_values(by='Average',ascending=False)
print(results_df.to_markdown(index=False))

| Model               |   Price Accuracy (%) |   Range Accuracy (%) |   Average |
|:--------------------|---------------------:|---------------------:|----------:|
| XGBoost             |                96.05 |                88.16 |     92.11 |
| Gradient Boosting   |                94.74 |                87.28 |     91.01 |
| Extra Trees         |                94.74 |                86.4  |     90.57 |
| Random Forest       |                92.11 |                86.4  |     89.25 |
| SVM (SVC)           |                90.79 |                84.65 |     87.72 |
| Decision Tree       |                89.91 |                85.09 |     87.5  |
| KNN                 |                89.47 |                82.46 |     85.96 |
| Logistic Regression |                86.4  |                79.39 |     82.89 |
